# Brain Stroke Classification with Decision Tree

Notebook này được viết lại để bám sát yêu cầu Lab 3 về **Decision Tree Modeling and Improvement**.

Điểm tham khảo từ notebook cũ của nhóm:
- Giữ lại tinh thần khám phá dữ liệu ban đầu.
- Tận dụng các quan sát đã có về `age`, `avg_glucose_level`, `bmi` và tình trạng mất cân bằng lớp.
- Viết lại theo flow dễ nộp báo cáo hơn: **tiền xử lý -> baseline tree -> phân tích cây -> cải tiến -> so sánh**.

Mục tiêu chính:
1. Huấn luyện một **baseline decision tree**.
2. Phân tích cấu trúc cây theo đúng yêu cầu phần **b, c**.
3. Thử thêm các cách cải tiến để nhóm tiện dùng cho phần **d, e** trong báo cáo.


## Bước 1. Khai báo thư viện và cấu hình chung

Ghi chú:
- `DecisionTreeClassifier` là mô hình chính của bài lab.
- `plot_tree` và `export_text` giúp trực quan hóa và giải thích luật tách của cây.
- `random_state=42` giúp kết quả có tính lặp lại.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = Path("brain_stroke.csv")

print(f"Data file exists: {DATA_PATH.exists()}")


## Bước 2. Đọc dữ liệu và mô tả dữ liệu

Mục tiêu của bước này:
- Xác nhận dataset đọc được bình thường.
- Kiểm tra số dòng, số cột, kiểu dữ liệu.
- Chuẩn bị thông tin cho phần mô tả dataset trong báo cáo.


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Kích thước dữ liệu:", df.shape)
print("Các cột:", list(df.columns))
print("\n5 dòng đầu tiên:")
display(df.head())


In [ ]:
print("Thông tin kiểu dữ liệu:")
df.info()

print("\nThống kê mô tả cho các cột số:")
display(df.describe().T)


## Bước 3. Kiểm tra chất lượng dữ liệu

Trong notebook cũ, nhóm đã kiểm tra dữ liệu thiếu và dữ liệu trùng.
Ở đây mình giữ lại bước đó nhưng trình bày gọn và rõ hơn để dễ đưa vào báo cáo.


In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
duplicate_count = df.duplicated().sum()
class_distribution = df["stroke"].value_counts().sort_index()
class_ratio = (df["stroke"].value_counts(normalize=True).sort_index() * 100).round(2)

print("Số lượng giá trị thiếu theo từng cột:")
display(missing_summary.to_frame(name="missing_count"))

print(f"Số dòng bị trùng: {duplicate_count}")

print("\nPhân bố biến mục tiêu `stroke`:")
summary_df = pd.DataFrame({
    "count": class_distribution,
    "ratio_percent": class_ratio,
})
display(summary_df)


## Bước 4. Khám phá nhanh dữ liệu theo hướng notebook cũ

Notebook cũ cho thấy:
- `age` có liên hệ đáng chú ý với đột quỵ.
- Dataset bị **mất cân bằng lớp** khá mạnh.
- `avg_glucose_level` và `bmi` cũng là các đặc trưng đáng quan tâm.

Ở đây mình giữ lại các biểu đồ chính để làm phần mô tả trực quan.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Biểu đồ 1: phân bố số lượng giữa 2 lớp.
sns.countplot(data=df, x="stroke", palette="Set2", ax=axes[0, 0])
axes[0, 0].set_title("Phân bố biến mục tiêu stroke")
axes[0, 0].set_xlabel("Stroke")
axes[0, 0].set_ylabel("Số lượng mẫu")

# Biểu đồ 2: tuổi theo trạng thái stroke.
sns.histplot(data=df, x="age", hue="stroke", bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Phân bố tuổi theo stroke")

# Biểu đồ 3: mức glucose trung bình theo stroke.
sns.histplot(data=df, x="avg_glucose_level", hue="stroke", bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Glucose theo stroke")

# Biểu đồ 4: BMI theo stroke.
sns.histplot(data=df, x="bmi", hue="stroke", bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("BMI theo stroke")

plt.tight_layout()
plt.show()


## Bước 5. Tiền xử lý dữ liệu cho Baseline Decision Tree

Các quyết định tiền xử lý ở bước baseline:
- Điền giá trị trung bình cho `bmi` nếu có thiếu.
- One-hot encoding cho các cột phân loại.
- **Không scale dữ liệu** vì Decision Tree không nhạy với thang đo như Logistic Regression hay SVM.
- **Không dùng SMOTE ở baseline** để baseline phản ánh mô hình gốc trung thực hơn.


In [ ]:
df_model = df.copy()

# Bổ sung lớp bảo vệ cho trường hợp dataset có giá trị thiếu ở cột BMI.
if df_model["bmi"].isna().sum() > 0:
    df_model["bmi"] = df_model["bmi"].fillna(df_model["bmi"].mean())

# Xác định các cột phân loại để mã hóa one-hot.
categorical_columns = df_model.select_dtypes(include="object").columns.tolist()
print("Các cột phân loại sẽ được mã hóa:", categorical_columns)

df_encoded = pd.get_dummies(df_model, columns=categorical_columns, drop_first=True)

# Tách đặc trưng đầu vào X và nhãn mục tiêu y.
X = df_encoded.drop(columns=["stroke"])
y = df_encoded["stroke"]

# Chia dữ liệu train/test có stratify để giữ tỷ lệ lớp tương đối ổn định.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Kích thước X_train:", X_train.shape)
print("Kích thước X_test :", X_test.shape)
print("Tỷ lệ stroke ở train:", round(y_train.mean() * 100, 2), "%")
print("Tỷ lệ stroke ở test :", round(y_test.mean() * 100, 2), "%")


## Bước 6. Tạo các hàm hỗ trợ đánh giá mô hình

Tách các hàm dùng chung giúp notebook dễ đọc hơn:
- `evaluate_tree_model`: huấn luyện và trả về toàn bộ chỉ số cần thiết.
- `show_confusion_matrix`: vẽ confusion matrix.
- `summarize_overfitting`: hỗ trợ nhận xét cây có đang học quá mức hay không.


In [ ]:
def evaluate_tree_model(model_name, model, X_train, X_test, y_train, y_test):
    # Huấn luyện mô hình trên tập train.
    model.fit(X_train, y_train)

    # Dự đoán nhãn và xác suất trên tập test.
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Gom tất cả chỉ số cần cho báo cáo vào một dictionary.
    result = {
        "model_name": model_name,
        "model": model,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
        "accuracy": accuracy_score(y_test, y_pred),
        "error_rate": 1 - accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "depth": model.get_depth(),
        "n_leaves": model.get_n_leaves(),
    }
    return result


def show_confusion_matrix(cm, title):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Stroke", "Stroke"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(title)
    plt.grid(False)
    plt.show()


def summarize_overfitting(train_acc, test_acc):
    gap = train_acc - test_acc
    if gap >= 0.10:
        return f"Train accuracy cao hơn test {gap:.4f} -> cây có dấu hiệu overfitting khá rõ."
    if gap >= 0.05:
        return f"Chênh lệch train/test là {gap:.4f} -> có dấu hiệu overfitting nhẹ."
    return f"Chênh lệch train/test là {gap:.4f} -> mức độ overfitting chưa đáng lo."


def result_to_frame(result):
    return pd.DataFrame(
        {
            "Metric": ["Train Accuracy", "Accuracy", "Error Rate", "Precision", "Recall", "F1-Score", "ROC-AUC", "Tree Depth", "Number of Leaves"],
            "Value": [
                result["train_accuracy"],
                result["accuracy"],
                result["error_rate"],
                result["precision"],
                result["recall"],
                result["f1_score"],
                result["roc_auc"],
                result["depth"],
                result["n_leaves"],
            ],
        }
    )


## Bước 7. Huấn luyện Baseline Decision Tree

Đây là mô hình cơ sở:
- Dùng `DecisionTreeClassifier` mặc định.
- Chưa chỉnh `class_weight`.
- Chưa cắt tỉa cây.
- Chưa đổi `criterion`.


In [ ]:
baseline_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
baseline_result = evaluate_tree_model(
    "Baseline Decision Tree",
    baseline_model,
    X_train,
    X_test,
    y_train,
    y_test,
)

print("=== KẾT QUẢ BASELINE MODEL ===")
display(result_to_frame(baseline_result))

print("Classification report:")
print(classification_report(y_test, baseline_result["y_pred"], zero_division=0))


In [ ]:
# Vẽ confusion matrix để xem mô hình phân loại sai ở đâu.
show_confusion_matrix(
    baseline_result["confusion_matrix"],
    "Confusion Matrix - Baseline Decision Tree"
)

# Vẽ 3 tầng đầu tiên của cây để dễ quan sát và chèn vào báo cáo.
plt.figure(figsize=(24, 12))
plot_tree(
    baseline_result["model"],
    feature_names=X.columns,
    class_names=["No Stroke", "Stroke"],
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3,
)
plt.title("Baseline Decision Tree - Top 3 Levels")
plt.show()


## Bước 8. Phân tích cây quyết định tạo ra (đúng trọng tâm phần c)

Phần này trả lời trực tiếp các ý:
- Cây sâu hay nông?
- Những thuộc tính nào quan trọng?
- Các luật tách chính là gì?
- Có dấu hiệu overfitting không?


In [ ]:
baseline_tree = baseline_result["model"]
root_feature_index = baseline_tree.tree_.feature[0]
root_threshold = baseline_tree.tree_.threshold[0]
root_feature_name = X.columns[root_feature_index]

feature_importance = (
    pd.Series(baseline_tree.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(10)
)

print("=== PHÂN TÍCH CÂY BASELINE ===")
print(f"Độ sâu của cây: {baseline_result['depth']}")
print(f"Số lượng lá: {baseline_result['n_leaves']}")
print(f"Nút gốc tách theo: {root_feature_name} <= {root_threshold:.2f}")
print(summarize_overfitting(baseline_result["train_accuracy"], baseline_result["accuracy"]))

print("\n10 đặc trưng quan trọng nhất:")
display(feature_importance.to_frame(name="importance"))

print("\nLuật tách rút gọn ở 4 tầng đầu:")
print(export_text(baseline_tree, feature_names=list(X.columns), max_depth=4))


In [ ]:
print("=== NHẬN XÉT GỢI Ý CHO BÁO CÁO ===")

top3 = feature_importance.head(3).index.tolist()

print(f"1. Cây baseline học rất sâu ({baseline_result['depth']} tầng) và có {baseline_result['n_leaves']} lá.")
print("2. Train accuracy đạt rất cao trong khi test accuracy thấp hơn đáng kể, nên có khả năng overfitting.")
print(f"3. Các thuộc tính quan trọng nhất của cây hiện tại là: {', '.join(top3)}.")
print("4. Quan sát ở các nhánh đầu cho thấy tuổi, glucose và BMI thường xuất hiện sớm trong các quyết định tách.")
print("5. Với dữ liệu mất cân bằng mạnh, accuracy có thể cao nhưng mô hình vẫn bỏ sót nhiều ca stroke.")


## Bước 9. Cải tiến mô hình Decision Tree

Dù nhóm bạn đang ưu tiên phần b, c, mình để luôn phần cải tiến để notebook đủ dùng cho báo cáo:

- **Improvement 1 - Class Weight**: ép mô hình chú ý hơn đến lớp thiểu số.
- **Improvement 2 - Pruning**: giới hạn độ sâu cây và số mẫu ở lá để giảm overfitting.
- **Improvement 3 - Entropy + Pruning**: đổi tiêu chí chia nhánh sang `entropy` và vẫn cắt tỉa cây.


In [ ]:
model_candidates = {
    "Baseline": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Class Weight": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "Pruning": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=10,
    ),
    "Entropy + Pruning": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        criterion="entropy",
        max_depth=5,
        min_samples_leaf=10,
    ),
}

all_results = {}

for model_name, model in model_candidates.items():
    all_results[model_name] = evaluate_tree_model(
        model_name,
        model,
        X_train,
        X_test,
        y_train,
        y_test,
    )

comparison_rows = []
for model_name, result in all_results.items():
    comparison_rows.append(
        {
            "Model": model_name,
            "Train Accuracy": result["train_accuracy"],
            "Accuracy": result["accuracy"],
            "Error Rate": result["error_rate"],
            "Precision": result["precision"],
            "Recall": result["recall"],
            "F1-Score": result["f1_score"],
            "ROC-AUC": result["roc_auc"],
            "Depth": result["depth"],
            "Leaves": result["n_leaves"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values(by="Accuracy", ascending=False)
display(comparison_df.round(4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.barplot(data=comparison_df, x="Accuracy", y="Model", palette="viridis", ax=axes[0, 0])
axes[0, 0].set_title("So sánh Accuracy")

sns.barplot(data=comparison_df, x="Recall", y="Model", palette="magma", ax=axes[0, 1])
axes[0, 1].set_title("So sánh Recall")

sns.barplot(data=comparison_df, x="F1-Score", y="Model", palette="crest", ax=axes[1, 0])
axes[1, 0].set_title("So sánh F1-Score")

sns.barplot(data=comparison_df, x="ROC-AUC", y="Model", palette="rocket", ax=axes[1, 1])
axes[1, 1].set_title("So sánh ROC-AUC")

plt.tight_layout()
plt.show()


In [ ]:
best_accuracy_model = comparison_df.sort_values(by="Accuracy", ascending=False).iloc[0]["Model"]
best_f1_model = comparison_df.sort_values(by="F1-Score", ascending=False).iloc[0]["Model"]
best_recall_model = comparison_df.sort_values(by="Recall", ascending=False).iloc[0]["Model"]

print("=== KẾT LUẬN TỪ PHẦN SO SÁNH ===")
print(f"Mô hình có Accuracy cao nhất: {best_accuracy_model}")
print(f"Mô hình có F1-Score cao nhất: {best_f1_model}")
print(f"Mô hình có Recall cao nhất: {best_recall_model}")
print("\nGợi ý viết báo cáo:")
print("- Nếu nhóm ưu tiên phát hiện được nhiều ca stroke hơn, hãy chú ý Recall và ROC-AUC.")
print("- Nếu nhóm ưu tiên độ chính xác tổng thể, Accuracy có thể là tiêu chí chính.")
print("- Với dữ liệu mất cân bằng, không nên nhìn mỗi Accuracy mà bỏ qua Recall/F1.")


## Bước 10. Trực quan cây sau cải tiến để đối chiếu với baseline

Cell này giúp nhóm có hình cây gọn hơn để đưa vào báo cáo và dễ bình luận vì sao pruning làm cây bớt phức tạp.


In [ ]:
improved_tree_result = all_results["Entropy + Pruning"]

show_confusion_matrix(
    improved_tree_result["confusion_matrix"],
    "Confusion Matrix - Entropy + Pruning"
)

plt.figure(figsize=(22, 10))
plot_tree(
    improved_tree_result["model"],
    feature_names=X.columns,
    class_names=["No Stroke", "Stroke"],
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3,
)
plt.title("Improved Decision Tree - Entropy + Pruning (Top 3 Levels)")
plt.show()


## Gợi ý kết luận cuối notebook

Các ý có thể dùng để viết nhận xét:
- Baseline tree cho accuracy khá cao nhưng cây rất sâu và có dấu hiệu overfitting.
- Khi dữ liệu mất cân bằng, các mô hình cải tiến có thể giảm accuracy nhưng tăng recall hoặc F1.
- `Entropy + Pruning` thường cho cấu trúc cây gọn hơn và cân bằng chỉ số tốt hơn baseline.
- Với bài toán stroke, cần cân nhắc giữa **accuracy** và khả năng **phát hiện ca dương tính**.
